# Bellman optimality: $V^*$, $Q^*$, and the greedy policy

_Reinforcement Learning_

**The best a state can ever be worth — and the one equation every control algorithm is trying to solve.**

Every state has a ceiling: the most discounted future reward you could ever collect starting there, if you played perfectly. That ceiling is the optimal state value $V^*(s)$. The matching quantity for a state-action pair — "commit to action $a$ now, then play perfectly" — is the optimal action value $Q^*(s,a)$.

       The whole point: if you knew $Q^*$, you would not need to plan at all. In any state you would just look at $Q^*(s,\cdot)$ and take the action with the largest value. That is the central result of this lesson — a policy that is greedy with respect to $Q^*$ is optimal. Finding $Q^*$ is solving the MDP.

---

This notebook is a practice scaffold for the **Bellman optimality: $V^*$, $Q^*$, and the greedy policy** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — numpy

### Step 1 — Build a tiny MDP

We describe a 3-state chain: `s0`, `s1`, and an absorbing `GOAL`. Two actions are available — `0` ("back") and `1` ("advance"). The transition tensor `P[s, a, s']` holds the probability of landing in `s'`, and the reward tensor `R[s, a, s']` gives the reward on that transition. Only entering `GOAL` pays off (+1), and `GOAL` loops back to itself forever.

In [ ]:
# Tiny MDP: 3 states (s0, s1, GOAL=2), 2 actions (0 = "back", 1 = "advance").
# advance: s0 -> s1 -> GOAL (absorbing). Reward +1 only when entering GOAL. gamma = 0.9.
nS = 3        # number of states
nA = 2        # number of actions
GOAL = 2      # index of the absorbing goal state
gamma = 0.9   # discount factor

P = np.zeros((nS, nA, nS))          # P[s, a, s'] = transition probability
P[0, 0, 0] = 1.0                    # s0, back   -> s0
P[0, 1, 1] = 1.0                    # s0, advance-> s1
P[1, 0, 0] = 1.0                    # s1, back   -> s0
P[1, 1, 2] = 1.0                    # s1, advance-> GOAL
P[2, 0, 2] = 1.0                    # GOAL is absorbing under either action
P[2, 1, 2] = 1.0

R = np.zeros((nS, nA, nS))          # R[s, a, s'] = reward on that transition
R[1, 1, 2] = 1.0                    # +1 for reaching GOAL

print("states:", nS, " actions:", nA, " gamma:", gamma)

### Step 2 — Solve for $Q^*$ by value iteration

Value iteration repeatedly applies the Bellman optimality update $Q^*(s,a) = \sum_{s'} P\,[R + \gamma\,\max_{a'} Q^*(s',a')]$. Each sweep first reads off the current state values $V^*(s') = \max_{a'} Q^*(s',a')$, then backs them up through the transition/reward tensors. The loop stops once an update changes `Q` by essentially nothing — that fixed point *is* $Q^*$.

In [ ]:
# Value iteration: Q*(s,a) = sum_s' P[R + gamma * max_a' Q*(s',a')]
Q = np.zeros((nS, nA))
for _ in range(500):
    V = Q.max(axis=1)                               # V*(s') = max_a' Q*(s',a')
    backup = R + gamma * V[None, None, :]           # R + gamma V*(s') for every (s,a,s')
    Q_new = np.einsum("sap,sap->sa", P, backup)     # expectation over s' under P
    if np.max(np.abs(Q_new - Q)) < 1e-12:           # converged to the fixed point
        break
    Q = Q_new

np.set_printoptions(precision=4, suppress=True)
print("Q* =\n", Q)                                 # [[0.81 0.9 ] [0.81 1.  ] [0.  0. ]]
print("V* = max_a Q* =", Q.max(axis=1))             # [0.9 1.  0. ]

### Step 3 — Verify the equation and read off the greedy policy

A correct $Q^*$ must satisfy the Bellman optimality equation *exactly*: plugging $V^* = \max_a Q^*$ back into the right-hand side should reproduce `Q` with zero residual. Once verified, the optimal policy falls out for free — just take $\pi^*(s) = \arg\max_a Q^*(s,a)$ in every state. No further planning is needed.

In [ ]:
# VERIFY the Bellman optimality equation: known Q* must satisfy it exactly.
Vstar = Q.max(axis=1)
backup = R + gamma * Vstar[None, None, :]
rhs = np.einsum("sap,sap->sa", P, backup)
print("max |Q* - RHS| =", np.max(np.abs(Q - rhs)))  # 0.0  -> the equation holds

# Extract the greedy optimal policy pi*(s) = argmax_a Q*(s,a).
pi = Q.argmax(axis=1)
print("pi*(s) =", pi)                               # [1 1 0] -> advance, advance, (GOAL: either)

## Visualize the data & results

_On a real 3x4 gridworld with a +1 goal and a -1 hazard, what is the optimal value V*(s) of every cell, and which way does the greedy optimal policy point?_

### Step 1 — Describe a 3x4 gridworld

This is the classic Russell & Norvig gridworld. The agent starts somewhere on a 3-row, 4-column grid with a `+1` goal, a `-1` hazard (both absorbing), and a blocked wall cell. Movement is **stochastic**: the intended direction happens with probability 0.8, and the agent slips perpendicular with probability 0.1 each way. Every non-terminal step costs `-0.04`, which nudges the optimal policy toward short paths.

In [ ]:
# 3x4 gridworld. Goal (0,3)=+1 (absorbing), hazard (1,3)=-1 (absorbing), wall (1,1) blocked.
# Stochastic moves: intended 0.8, each perpendicular slip 0.1. step cost -0.04, gamma 0.9.
ROWS, COLS = 3, 4
WALL, GOAL, HAZARD = (1, 1), (0, 3), (1, 3)
gamma, step_cost = 0.9, -0.04
acts = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}

def ok(s):                                        # is cell s on the grid and not the wall?
    return 0 <= s[0] < ROWS and 0 <= s[1] < COLS and s != WALL

states = [(r, c) for r in range(ROWS) for c in range(COLS) if (r, c) != WALL]
terminals = {GOAL, HAZARD}

def reward(s):                                    # reward for being in state s
    return 1.0 if s == GOAL else (-1.0 if s == HAZARD else step_cost)

### Step 2 — Model the stochastic transitions

For a chosen action we build $P(s' \mid s, a)$ as a dictionary. With probability 0.8 the agent moves as intended; with 0.1 each it slips to one of the two perpendicular directions. Any move that would leave the grid or hit the wall leaves the agent where it was — so that probability mass piles back onto `s`.

In [ ]:
def trans(s, a):                                  # P(s' | s, a) as a dict
    if a in ("up", "down"):
        perp = ["left", "right"]                  # perpendicular directions to slip toward
    else:
        perp = ["up", "down"]
    outcomes = [(0.8, a), (0.1, perp[0]), (0.1, perp[1])]
    res = {}
    for p, mv in outcomes:
        d = acts[mv]
        ns = (s[0] + d[0], s[1] + d[1])
        if not ok(ns):                            # bumping a wall/edge keeps you put
            ns = s
        res[ns] = res.get(ns, 0.0) + p
    return res

### Step 3 — Value-iterate, then read $V^*$ and $\pi^*$

We sweep the Bellman optimality update $V^*(s) = R(s) + \gamma\,\max_a \sum_{s'} P(s'\mid s,a)\,V^*(s')$ across every non-terminal cell until the values stop changing. Terminal cells are pinned to their reward. Afterward we lay the converged values into a grid (the wall stays blank) and extract the greedy direction in each cell.

In [ ]:
# Value iteration: V*(s) = max_a sum_s' P(s'|s,a)[R + gamma V*(s')]
V = {s: 0.0 for s in states}
for sweep in range(1000):
    nV = {}
    delta = 0.0
    for s in states:
        if s in terminals:
            nV[s] = reward(s)                     # terminal value is just its reward
            continue
        action_values = [
            sum(p * V[ns] for ns, p in trans(s, a).items()) for a in acts
        ]
        nV[s] = reward(s) + gamma * max(action_values)
        delta = max(delta, abs(nV[s] - V[s]))
    V = nV
    if delta < 1e-9:                              # values stopped changing -> converged
        break
print("converged in", sweep + 1, "sweeps")        # 30

# Heatmap matrix of V* (wall = NaN / blank).
grid = np.full((ROWS, COLS), np.nan)
for s in states:
    grid[s] = V[s]
print(np.round(grid, 3))

# Greedy optimal policy pi*(s) = argmax_a Q*(s,a).
for s in sorted(states):
    if s in terminals:
        continue
    best = max(acts, key=lambda a: sum(p * V[ns] for ns, p in trans(s, a).items()))
    print(s, "->", best)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A state $s$ has two actions. Action $a_1$ is deterministic to a state with $V^*=5$; action $a_2$ goes with probability $0.5$ to a state with $V^*=12$ and probability $0.5$ to a state with $V^*=0$. Every transition gives reward $R=1$ and $\gamma=0.9$. Find $Q^*(s,a_1)$, $Q^*(s,a_2)$, $V^*(s)$, and $\pi^*(s)$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute $Q^*(s,a_1) = R + \gamma V^*(\text{next}) = 1 + 0.9\times 5$. — _$a_1$ is deterministic, so the expectation over $s'$ is a single term._
- Compute $Q^*(s,a_2) = R + \gamma\,[0.5\times 12 + 0.5\times 0]$. — _$a_2$ is stochastic, so average the next-state values with the transition probabilities $P(s'\mid s,a_2)$._
- Take $V^*(s) = \max(Q^*(s,a_1), Q^*(s,a_2))$ and $\pi^*(s) = \arg\max$. — _The Bellman optimality equation keeps the best action's value; the policy is the action achieving it._

**Answer:** $Q^*(s,a_1) = 1 + 0.9\times 5 = 5.5$. $Q^*(s,a_2) = 1 + 0.9\times(0.5\times12 + 0.5\times0) = 1 + 0.9\times 6 = 6.4$. So $V^*(s) = \max(5.5, 6.4) = 6.4$ and $\pi^*(s) = a_2$. The risky action wins because its expected next value ($6$) beats the safe $5$, even though it sometimes lands on the $0$ state.

</details>

**Problem 2.** In one sentence each: why can policy evaluation be solved by a single linear solve $V = (I-\gamma P)^{-1}R$, while the Bellman OPTIMALITY equation cannot — and what do we do instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look at the operator in each case: $\sum_a \pi(a\mid s)(\dots)$ for evaluation vs. $\max_a(\dots)$ for optimality. — _The form of the action aggregation decides linearity._
- Note that $\sum_a\pi$ is a weighted sum (linear in $V$) but $\max_a$ is not. — _A $\max$ has kinks; it is not a linear map, so no matrix inverse expresses the fixed point._

**Answer:** Policy evaluation averages over actions with fixed weights $\pi(a\mid s)$, which is linear in $V$, so the fixed point $V = R + \gamma P V$ rearranges to the closed-form solve $V = (I-\gamma P)^{-1}R$. The optimality equation replaces that average with $\max_a$, which is non-linear (a kink), so no single matrix inverse gives $V^*$. Instead we iterate the Bellman optimality operator $\mathcal{T}^*$ — value iteration — which is a $\gamma$-contraction and converges geometrically to the unique fixed point $V^*$.

</details>